
# eThekwini 2026 Local Government Election — Reproducible Machine-Learning Analysis

**Target date:** 4 November 2026  
**Study area:** eThekwini Metropolitan Municipality  
**Purpose:** estimate party vote shares, three ward leaders and voter turnout.

### Four required analytical outputs
1. Metro-wide largest projected party and mathematical coalition-threshold assessment.
2. Projected leading party in three geographically defensible 2026 wards.
3. Projected vote total and vote share for each relevant party.
4. Projected turnout as estimated voters and turnout rate.

### Evidence classification
- **HISTORICAL OBSERVATION:** directly downloaded from an official source.
- **DERIVED / ALIGNED HISTORICAL ESTIMATE:** calculated from historical observations, including geographic allocation.
- **2026 MODEL OUTPUT:** produced by a trained model using only information available before election day.

**Important:** The model does not predict who will govern eThekwini.



## 1. Methodological decisions

### Data
- IEC 2021 Municipal Election results at voting-district level.
- IEC 2024 National/Provincial Election results at voting-district level as a political-signal feature, not as a municipal result.
- Stats SA Census 2022 socioeconomic/demographic indicators.
- IEC/MDB 2021, 2024 and 2026 voting-district/ward geography.
- Latest available 2026 registered-voter counts.
- 2026 certified independent-candidate information where available.

### Geographic alignment
The 2026 ward geography is the prediction geography. Historical voting results are aligned through voting districts / spatial relationships. Ward numbers alone are never used as a cross-year key.

### Models
- Party vote share: Random Forest Regression + Linear Regression benchmark.
- Ward leader: Random Forest multiclass classification.
- Turnout rate: Gradient Boosting Regression + Random Forest benchmark.

### Validation
Use time-aware historical backtesting and hold-out evaluation. A random split is not the sole validation method because election observations are time ordered.

### Coalition output
Only a mathematical 50% threshold check is reported. Coalition negotiations and governing outcomes are not modelled.


In [ ]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path(".")
RAW = ROOT / "data" / "raw"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

print("Project root:", ROOT.resolve())
print("Raw data folder:", RAW.resolve())



## 2. Data acquisition and provenance

Download the raw files from the official sources in `data/raw/README_DATA.md`.

The IEC publishes election reports and supports municipality, ward and voting-district result levels. Its Atlas explicitly supports geospatial comparison of LGE and NPE results. Stats SA provides Census 2022 data and ward-level small-area estimates.

Do not overwrite raw files during cleaning. The raw directory is treated as immutable evidence.


In [ ]:

# Create a provenance template.
manifest = pd.DataFrame([
    ["IEC", "2021 Municipal Election", "https://www.elections.org.za/pw/Elections-and-results/Municipal-Elections-2021", ""],
    ["IEC", "Atlas of Results", "https://atlas.elections.org.za/", ""],
    ["IEC", "2024 NPE results", "https://www.elections.org.za/content/pages/reports/npe/selection.aspx", ""],
    ["Stats SA", "Census 2022", "https://isibaloweb.statssa.gov.za/pages/surveys/pss/censuses/census2022/census2022.php", ""],
    ["MDB", "Ward delimitation", "https://www.demarcation.org.za/", ""],
    ["eThekwini", "2026 ward delimitation statement", "https://durban.gov.za/press-statement/ethekwini-municipality-ward-delimitation-process-completed", ""],
], columns=["organisation","dataset","source_url","download_date"])
manifest.to_csv(OUT / "source_manifest.csv", index=False)
manifest


In [ ]:

required = [
    "lge2021_ethekwini_vd.csv",
    "npe2024_ethekwini_vd.csv",
    "registration_2021_2026.csv",
    "census2022_ward.csv",
]
missing = [f for f in required if not (RAW / f).exists()]
if missing:
    print("DATA ACQUISITION CHECK:")
    for f in missing:
        print("  Missing:", f)
    print("\nDownload the official files described in data/raw/README_DATA.md, standardise them to the required schema, then rerun this notebook.")
else:
    print("All minimum tabular input files are present.")



## 3. Load and validate the tabular inputs

The next cell intentionally fails fast when a required column is missing. This makes the notebook auditable rather than silently producing a wrong result.


In [ ]:

def load_csv(name):
    path = RAW / name
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return pd.read_csv(path)

def require_columns(df, cols, name):
    missing = sorted(set(cols) - set(df.columns))
    if missing:
        raise ValueError(f"{name}: missing required columns: {missing}")

if all((RAW / f).exists() for f in required):
    lge21 = load_csv("lge2021_ethekwini_vd.csv")
    npe24 = load_csv("npe2024_ethekwini_vd.csv")
    registration = load_csv("registration_2021_2026.csv")
    census = load_csv("census2022_ward.csv")

    require_columns(lge21, ["municipality_code","ward_2021","vd_2021","party","votes","valid_votes","registered_voters","votes_cast"], "lge21")
    require_columns(npe24, ["municipality_code","vd_2024","ballot","party","votes","valid_votes","registered_voters","votes_cast"], "npe24")
    require_columns(registration, ["municipality_code","ward_2026","registered_voters_2026"], "registration")
    require_columns(census, ["ward_source_id","population","unemployment_rate","secondary_or_higher_education_rate","electricity_access_rate","piped_water_access_rate"], "census")

    print(lge21.shape, npe24.shape, registration.shape, census.shape)
else:
    lge21 = npe24 = registration = census = None



## 4. Geography audit

**Do not continue to ward-level modelling until the 2021→2026 relationship has been checked.**

The Municipal Demarcation Board finalised 4,488 wards nationally for the 2026 elections. eThekwini's 2026 delimitation produces 112 wards, and multiple existing wards were affected. The notebook therefore requires an explicit crosswalk.

Recommended hierarchy:
1. Exact voting-district membership.
2. Spatial overlay with voter/population weights.
3. Aggregate historical units where necessary.
4. Mark unresolved relationships and exclude them from ward-level comparison.

The safest prediction geography is the 2026 ward.


In [ ]:

# Expected optional geographic crosswalk.
crosswalk_path = RAW / "vd_crosswalk_2021_2024_2026.csv"

if crosswalk_path.exists():
    crosswalk = pd.read_csv(crosswalk_path)
    required_cw = ["vd_2021","vd_2024","vd_2026","ward_2026","alignment_status","weight_2021_to_2026"]
    require_columns(crosswalk, required_cw, "geographic crosswalk")

    allowed = {"EXACT","AGGREGATED","DISAGGREGATED","UNRESOLVED"}
    bad = set(crosswalk["alignment_status"].dropna()) - allowed
    if bad:
        raise ValueError(f"Unknown alignment status values: {bad}")

    if (crosswalk["alignment_status"] == "UNRESOLVED").any():
        print("Warning: unresolved geographic relationships exist. They will be excluded from ward-level modelling.")

    crosswalk.to_csv(OUT / "geographic_crosswalk_audited.csv", index=False)
    print(crosswalk["alignment_status"].value_counts(dropna=False))
else:
    crosswalk = None
    print("No crosswalk supplied yet. Create it from IEC/MDB voting-district geography before ward modelling.")



### Geographic weighting rule

If a historical voting district is split across 2026 wards, historical votes are allocated using the best available registered-voter or population weight. Area-only weighting is a fallback and must be documented.

For an historical vote count `V`:

`aligned_votes = V × weight_2021_to_2026`

The aligned value is **not an observed 2026 vote**. It is a derived historical estimate.


In [ ]:

def weighted_alignment(results, cw, old_id, new_id, weight_col):
    x = results.merge(
        cw[[old_id, new_id, weight_col, "alignment_status"]],
        on=old_id, how="left"
    )
    x["aligned_votes"] = x["votes"] * x[weight_col]
    return x

if lge21 is not None and crosswalk is not None:
    aligned21 = weighted_alignment(
        lge21, crosswalk,
        "vd_2021", "vd_2026", "weight_2021_to_2026"
    )
    aligned21 = aligned21[aligned21["alignment_status"] != "UNRESOLVED"].copy()
    aligned21["data_status"] = "DERIVED / ALIGNED HISTORICAL ESTIMATE"
    print(aligned21.head())
else:
    aligned21 = None



## 5. Prepare historical party features

2021 municipal results are the main historical municipal outcome.

2024 NPE results are retained as a political signal. Because 2024 was not a municipal election, the notebook does not label 2024 party shares as municipal vote shares.


In [ ]:

def shares_from_long(df, geo_col, vote_col, prefix):
    pivot = df.pivot_table(
        index=geo_col,
        columns="party",
        values=vote_col,
        aggfunc="sum",
        fill_value=0
    )
    totals = pivot.sum(axis=1).replace(0, np.nan)
    share = pivot.div(totals, axis=0)
    share.columns = [f"{prefix}_{str(c).strip().upper()}_share" for c in share.columns]
    return share.reset_index(), pivot.reset_index()

if aligned21 is not None:
    party21, votes21 = shares_from_long(aligned21, "vd_2026", "aligned_votes", "LGE21")
else:
    party21 = votes21 = None

if npe24 is not None and crosswalk is not None:
    npe24a = npe24.merge(
        crosswalk[["vd_2024","vd_2026","alignment_status","weight_2024_to_2026"]],
        on="vd_2024", how="left"
    )
    npe24a = npe24a[npe24a["alignment_status"] != "UNRESOLVED"].copy()
    npe24a["aligned_votes"] = npe24a["votes"] * npe24a["weight_2024_to_2026"]
    # User must filter to one documented ballot.
    ballot_counts = npe24a["ballot"].value_counts()
    print("2024 ballot counts:", ballot_counts.to_dict())
else:
    npe24a = None



## 6. Party harmonisation

Party names differ across releases. Create an explicit mapping rather than relying on fuzzy matching.

Keep `MK` as a distinct 2024 signal because it did not contest the 2021 municipal election. Do not replace a genuine historical absence with zero without documenting that it means "not contested / not observed", not zero voter support.


In [ ]:

PARTY_MAP = {
    "AFRICAN NATIONAL CONGRESS": "ANC",
    "ANC": "ANC",
    "DEMOCRATIC ALLIANCE": "DA",
    "DA": "DA",
    "ECONOMIC FREEDOM FIGHTERS": "EFF",
    "EFF": "EFF",
    "INKATHA FREEDOM PARTY": "IFP",
    "IFP": "IFP",
    "UMKHONTO WESIZWE": "MK",
    "MK PARTY": "MK",
    "MK": "MK",
    "ACTIONSA": "ACTIONSA",
    "ACTIONSA": "ACTIONSA",
}

def harmonise_party(s):
    u = str(s).strip().upper()
    return PARTY_MAP.get(u, "OTHER")

if lge21 is not None:
    lge21["party_std"] = lge21["party"].map(harmonise_party)
if npe24 is not None:
    npe24["party_std"] = npe24["party"].map(harmonise_party)

print("Party mapping ready.")



## 7. Census preparation

Census variables are historical observations. They become model features only after being aligned to the 2026 ward geography.

If Stats SA's source geography is not the 2026 ward geography, use the same crosswalk/spatial allocation procedure and label the resulting values as derived estimates.


In [ ]:

census_numeric = [
    "population",
    "unemployment_rate",
    "secondary_or_higher_education_rate",
    "electricity_access_rate",
    "piped_water_access_rate"
]

if census is not None:
    census[census_numeric] = census[census_numeric].apply(pd.to_numeric, errors="coerce")
    census[census_numeric] = census[census_numeric].fillna(census[census_numeric].median())
    print(census[census_numeric].describe().T)



## 8. Build the modelling table

The exact implementation depends on the supplied geographic crosswalk. The final table must contain one row per 2026 ward and only variables available before 4 November 2026.

The target columns shown below are placeholders for historical backtesting. They are never filled with actual 2026 results before election day.


In [ ]:

# Example feature names after harmonisation.
candidate_features = [
    "ANC_2021_share", "DA_2021_share", "EFF_2021_share", "IFP_2021_share",
    "ANC_2024_signal_share", "DA_2024_signal_share", "MK_2024_signal_share",
    "EFF_2024_signal_share", "IFP_2024_signal_share",
    "turnout_2021", "registered_voters_2026",
    "population", "unemployment_rate",
    "secondary_or_higher_education_rate",
    "electricity_access_rate", "piped_water_access_rate",
    "independent_candidate_count"
]
print("Candidate feature list:", candidate_features)



## 9. Historical derived features

Examples:
- party change between elections, explicitly labelled as an electoral-signal change;
- historical turnout change;
- registration growth;
- demographic ratios.

Do not call the 2021→2024 difference a "municipal swing", because 2024 was a national/provincial election.


In [ ]:

def add_safe_changes(df):
    if {"ANC_2021_share","ANC_2024_signal_share"}.issubset(df.columns):
        df["ANC_electoral_signal_change"] = df["ANC_2024_signal_share"] - df["ANC_2021_share"]
    if {"turnout_2021","turnout_2024_signal"}.issubset(df.columns):
        df["turnout_signal_change"] = df["turnout_2024_signal"] - df["turnout_2021"]
    return df



## 10. Time-aware validation strategy

Do not use a random split as the only test.

Use historical backtesting:
- Fit on information available before a historical election.
- Evaluate on the later election.
- Keep the election target appropriate to the election type.
- Use 2024 as a political-signal validation source, not as a municipal target.

For the final 2026 model, all features must have a data timestamp no later than election day.


In [ ]:

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }

def evaluate_regressors(X_train, y_train, X_test, y_test):
    models = {
        "Linear Regression": Pipeline([
            ("scale", StandardScaler()),
            ("model", LinearRegression())
        ]),
        "Random Forest": RandomForestRegressor(
            n_estimators=500, max_depth=8, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        )
    }
    rows = []
    fitted = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        m = regression_metrics(y_test, pred)
        rows.append({"Model": name, **m})
        fitted[name] = model
    return pd.DataFrame(rows), fitted



## 11. Party-share modelling

Party share is continuous, so use regression.

The Random Forest is the main non-linear model and Linear Regression is a transparent benchmark. Select the model using historical validation metrics, not by choosing the model that gives the most favourable 2026 result.


In [ ]:

# Example function for a single party target.
def fit_party_model(df, target, features, train_mask, test_mask):
    use = df[features + [target]].dropna()
    train = use.loc[train_mask.loc[use.index]]
    test = use.loc[test_mask.loc[use.index]]

    X_train, y_train = train[features], train[target]
    X_test, y_test = test[features], test[target]

    scores, fitted = evaluate_regressors(X_train, y_train, X_test, y_test)
    return scores, fitted



## 12. Ward-leader classification

The target is categorical: ANC, DA, MK, EFF, IFP, Other.

Use Random Forest classification. Evaluate accuracy, balanced accuracy and the confusion matrix. Because class sizes can be unequal, balanced accuracy is reported alongside ordinary accuracy.


In [ ]:

def fit_ward_classifier(X_train, y_train, X_test, y_test):
    clf = RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    metrics = {
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, pred)
    }
    return clf, metrics, pred



## 13. Turnout modelling

Turnout rate is a continuous target between 0 and 1.

Use Gradient Boosting as the main model and Random Forest as a benchmark. After predicting turnout rate, convert it into an estimated number of voters using the latest available 2026 registered-voter count.


In [ ]:

def fit_turnout_model(X_train, y_train, X_test, y_test):
    models = {
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=300, learning_rate=0.03, max_depth=2,
            random_state=42
        ),
        "Random Forest": RandomForestRegressor(
            n_estimators=500, max_depth=8, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        )
    }
    rows, fitted = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = np.clip(model.predict(X_test), 0, 1)
        m = regression_metrics(y_test, pred)
        rows.append({"Model": name, **m})
        fitted[name] = model
    return pd.DataFrame(rows), fitted



## 14. Three-ward selection

Select the wards only after geographic alignment.

The selection is reproducible:
1. `stable`: strongest historical top-party margin with a defensible 2021→2026 mapping.
2. `competitive`: smallest top-two historical margin.
3. `transition`: largest defensible change in electoral signal.

Exclude wards with unresolved geographic alignment.


In [ ]:

def select_three_wards(ward_summary):
    eligible = ward_summary[ward_summary["alignment_ok"]].copy()

    stable = eligible.sort_values("historical_margin", ascending=False).iloc[0]
    competitive = eligible.sort_values("historical_margin", ascending=True).iloc[0]
    transition = eligible.sort_values("political_signal_change", ascending=False).iloc[0]

    selected = pd.DataFrame([
        {"selection_type": "Stable stronghold", **stable.to_dict()},
        {"selection_type": "Competitive", **competitive.to_dict()},
        {"selection_type": "Political transition", **transition.to_dict()},
    ])
    return selected



## 15. Final 2026 output construction

### Party shares
Predict each relevant party's share, clip negatives, then normalise so shares sum to 100%.

### Party votes
`projected_votes = projected_share × projected_valid_votes`

### Turnout
`projected_voters = projected_turnout_rate × registered_voters_2026`

### Coalition threshold
Report only whether the largest projected party crosses 50%. Do not infer governing coalitions.


In [ ]:

def normalise_party_predictions(raw_dict):
    names = list(raw_dict.keys())
    raw = np.array([max(float(raw_dict[n]), 0.0) for n in names])
    if raw.sum() == 0:
        shares = np.repeat(1 / len(raw), len(raw))
    else:
        shares = raw / raw.sum()
    return pd.DataFrame({
        "party": names,
        "projected_share": shares,
        "status": "2026 MODEL OUTPUT"
    })

def coalition_threshold(projected):
    row = projected.sort_values("projected_share", ascending=False).iloc[0]
    return {
        "largest_party": row["party"],
        "largest_share": row["projected_share"],
        "mathematical_50pct_threshold_reached": bool(row["projected_share"] >= 0.50),
        "interpretation": "Vote-share threshold only; governing coalition is not modelled."
    }



## 16. Required output tables

The notebook should export:
- `outputs/metro_party_projection.csv`
- `outputs/ward_projection.csv`
- `outputs/turnout_projection.csv`
- `outputs/model_evaluation.csv`
- `outputs/selected_wards.csv`
- `outputs/geographic_crosswalk_audited.csv`

The dashboard reads these files.


In [ ]:

# Output helper.
def save_output(df, filename):
    path = OUT / filename
    df.to_csv(path, index=False)
    print("Saved:", path)

# Call save_output(...) after the final modelling cells have been populated.



## 17. Visualisation plan

The final notebook should contain:
1. 2021 vs 2026 ward geography map.
2. Historical party-share comparison.
3. Historical turnout trend.
4. Model actual-vs-estimated plot.
5. Feature-importance plot.
6. 2026 metro projected party shares.
7. Three selected ward projections.
8. 2026 projected turnout.

Every chart must state whether its values are historical, derived, or model output.


In [ ]:

def plot_projected_party_shares(projection):
    p = projection.sort_values("projected_share", ascending=True)
    plt.figure(figsize=(10, 6))
    plt.barh(p["party"], p["projected_share"] * 100)
    plt.xlabel("Projected vote share (%)")
    plt.title("eThekwini 2026 — Model Output Only")
    plt.tight_layout()
    plt.show()



# 18. Final reporting checklist

Before submission confirm:

- [ ] Raw sources are official and recorded.
- [ ] Raw files are unchanged.
- [ ] Geographic crosswalk is included.
- [ ] Changed wards are explicitly documented.
- [ ] 2024 NPE data are not described as 2024 municipal votes.
- [ ] No post-election 2026 results are used as features.
- [ ] Historical observations are separated from derived estimates.
- [ ] All 2026 values are labelled model output.
- [ ] Party model is evaluated with MAE/RMSE/R².
- [ ] Ward classifier is evaluated with accuracy/balanced accuracy/confusion matrix.
- [ ] Turnout model is evaluated with MAE/RMSE/R².
- [ ] Three wards are selected using an explicit rule.
- [ ] The largest projected party is reported without saying it will govern.
- [ ] Coalition output is limited to a mathematical threshold assessment.
- [ ] The dashboard reproduces the notebook's final outputs.
